# Embeddings and Vector Databases with Pinecone

**Embeddings** are vector representations of text. They are used to represent text in a vector space, where the distance between vectors represents the semantic similarity between the texts.

**Vector Databases** are databases that store vectors and their associated metadata. They are used to store and retrieve embeddings. Vector databases are organized into indexes (also called namespaces) - similar to tables in a relational database.

[Pinecone](https://www.pinecone.io/) is a vector database service that allows you to store and retrieve embeddings. It is a hosted service that allows you to scale your vector database as needed.

There are two main ways to use Pinecone:

1. **Store an embedding** - Store an embedding in a vector database.
    - Embed the text you want to store.
    - Create a document with the embedding and metadata.
    - Store the document in a vector database.
2. **Query a vector database** - Query a vector database for the most similar embeddings to a given query.
    - Embed the query.
    - Query the vector database with the embedded query.
    - Retrieve the most similar embeddings to the query.

### Install Pinecone and OpenAI

In [17]:
!pip install pinecone openai


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Import Libraries

In [18]:
import os
import uuid
from datetime import datetime, timezone
from pinecone import Pinecone
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

### Define Environment variables

In [19]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY") # Pinecone API key
PINECONE_INDEX_NAME=os.getenv("PINECONE_INDEX_NAME") # Name of the vector database index
PINECONE_NAMESPACE=os.getenv("PINECONE_NAMESPACE") # Namespace in your index on Pinecone.io
EMBEDDING_MODEL=os.getenv("EMBEDDING_MODEL") # OpenAI embedding model
EMBEDDING_DIMENSIONS=os.getenv("EMBEDDING_DIMENSIONS") # Number of dimensions in the embedding

### Initialize Pinecone and OpenAI

In [20]:
# Initialize Pinecone for vector database
pc = Pinecone(PINECONE_API_KEY)
# Initialize the vector database index
index = pc.Index(PINECONE_INDEX_NAME)
# Initialize OpenAI for embeddings 
client = OpenAI()

## 1. Store an embedding
-----------

In [21]:
string_to_store = "My favorite fruit is mango."

### Embed the string

In [22]:
# OpenAI embeddings
def get_embeddings(string_to_embed):
    response = client.embeddings.create(
        input=string_to_embed,
        model=EMBEDDING_MODEL,
        dimensions=int(EMBEDDING_DIMENSIONS)
    )
    return response.data[0].embedding

In [23]:
vector = get_embeddings(string_to_store)

In [24]:
print(f"Vector representation of {string_to_store}: \n", vector)

Vector representation of My favorite fruit is mango.: 
 [0.04701041802763939, -0.040899477899074554, -0.03645705804228783, 0.03399599716067314, -0.024089189246296883, 0.025841129943728447, -0.009687816724181175, 0.059148866683244705, 0.004335532430559397, -0.002964221639558673, -0.004630129784345627, 0.012420009821653366, -0.022253822535276413, 0.0008661940228193998, -0.028114480897784233, 0.011481470428407192, 0.00470834132283926, 0.0015107884537428617, 0.012784997932612896, 0.005401818081736565, 0.02544485777616501, 0.03754159435629845, -0.013577542267739773, 0.03583136573433876, 0.033912573009729385, 0.0026565890293568373, -0.015120918862521648, 0.04425736516714096, 0.016372306272387505, -0.07128731161355972, 0.021982688456773758, -0.00017385798855684698, 0.004893442150205374, -0.04534190148115158, 0.023630347102880478, -0.02233724854886532, 0.01780097186565399, -0.002464970573782921, 0.015308626927435398, 0.0338917151093483, -0.02100243605673313, 0.040878623723983765, 0.02796848677

### Define the vector metadata to store in the vector database

In [25]:
user_id = "1234"
current_time = datetime.now(tz=timezone.utc)

### Build the vector document to be stored

In [26]:
# Build document dictionary
documents = [
    {
        "id": str(uuid.uuid4()),
        "values": vector,
        "metadata": {
            "user_id": user_id,
            "timestamp": str(current_time),
            "payload": string_to_store,
        },
    }
]


### Store the vector document in the vector database

In [27]:
index.upsert(
    namespace=PINECONE_NAMESPACE,
    vectors=documents,
)

{'upserted_count': 1}

## 2. Query a vector database
-----------

In [28]:
query_string = "What are my preferences?"
user_id = "1234"
top_k = 1 # This is the number of most similar embeddings to return

### Embed the query

In [29]:
vector = get_embeddings(query_string)

### Query the vector database for similar top_k embeddings + filters

In [30]:
response = index.query(
    vector=vector,
    filter={
        "user_id": {"$eq": user_id},
    },
    namespace=PINECONE_NAMESPACE,
    include_metadata=True,
    top_k=top_k,
)

In [31]:
response

{'matches': [{'id': 'adfb0430-a84d-42df-9a5a-131ae7999b51',
              'metadata': {'payload': 'My favorite fruit is mango.',
                           'timestamp': '2025-12-15 09:54:42.060155+00:00',
                           'user_id': '1234'},
              'score': 0.247634009,
              'values': []}],
 'namespace': 'https://memory-agent-2xdin0a.svc.aped-4627-b74a.pinecone.io',
 'usage': {'read_units': 1}}

### Build the memories list

In [32]:
memories = []
if matches := response.get("matches"):
    memories = [m["metadata"]["payload"] for m in matches]
memories

['My favorite fruit is mango.']